# NAIC WP7 UC7: Latent Representation of PDE Solutions

*Learning compact representations of PDE solution manifolds via autoencoders and cross-modal latent alignment.*

> **Key result:** Solutions of the 2D convection-diffusion equation, parameterized by divergence-free streamfunction coefficients, live on a low-dimensional manifold that can be faithfully captured by a compact latent space shared across grid resolutions.

**Note:** This notebook uses small synthetic examples (low sample counts, coarse grids) to illustrate the concepts interactively. For full-scale training with autoencoders and latent alignment, see `demonstrator_pde.ipynb`.

## 0. Setup

In [ ]:
import sys
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from matplotlib import cm

sys.path.insert(0, 'src/')
from cd2d_streamfunc import (
    make_grid,
    monomial_multiindices,
    sample_coeffs,
    eval_velocity_from_streamfunc_coeffs,
    solve_level,
    gaussian_all_boundaries,
)

# Reproducibility
np.random.seed(42)

print('Setup complete.')

## 1. The Convection-Diffusion Problem

We solve the **steady 2D convection-diffusion equation** on the square domain $[-1,1]^2$:

$$\nabla \cdot (\mathbf{v} u) - \nabla \cdot (D \nabla u) = 0$$

where:
- $u(x,y)$ is the scalar field (e.g., temperature or concentration),
- $D = 1$ is the constant diffusion coefficient,
- $\mathbf{v}(x,y)$ is a **divergence-free** advection velocity field,
- **Dirichlet boundary conditions** are prescribed on all four boundaries as Gaussian bumps.

The velocity field is derived from a polynomial **streamfunction** $\psi(x,y)$, ensuring $\nabla \cdot \mathbf{v} = 0$ by construction.

In [ ]:
# Visualize the Dirichlet boundary conditions
n = 32
g = gaussian_all_boundaries(n, sigma=0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Full BC field
im = axes[0].imshow(g.T, origin='lower', extent=[-1, 1, -1, 1], cmap='inferno')
axes[0].set_title('Dirichlet BCs: Gaussian bumps on all boundaries')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Profile along bottom boundary
x_grid = np.linspace(-1, 1, n)
axes[1].plot(x_grid, g[:, 0], 'r-', linewidth=2, label='Bottom (y=-1)')
axes[1].plot(x_grid, g[:, -1], 'b--', linewidth=2, label='Top (y=+1)')
axes[1].plot(x_grid, g[0, :], 'g-.', linewidth=2, label='Left (x=-1)')
axes[1].set_title('Boundary profiles')
axes[1].set_xlabel('Coordinate')
axes[1].set_ylabel('g(boundary)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Generating PDE Solutions

The advection velocity is derived from a polynomial streamfunction:

$$\psi(x,y) = \sum_{i+j \leq n_{\text{sf}},\, (i,j) \neq (0,0)} a_{ij}\, x^i y^j$$

The velocity components are:

$$v_x = \frac{\partial \psi}{\partial y}, \quad v_y = -\frac{\partial \psi}{\partial x}$$

This guarantees $\nabla \cdot \mathbf{v} = 0$ by construction. By sampling different coefficient vectors $\{a_{ij}\}$, we generate a **family of PDE solutions** with diverse flow patterns.

In [ ]:
# Generate 4 solutions with different streamfunction coefficients
n_sf = 4
n_grid = 32
vel_scale = 1e4

C, idx = sample_coeffs(n_sf, 4, mode='uniform', seed=42)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes_flat = axes.ravel()

for k in range(4):
    u = solve_level(n_grid, C[k], idx, vel_scale)
    im = axes_flat[k].imshow(
        u.T, origin='lower', extent=[-1, 1, -1, 1],
        cmap='viridis', aspect='equal',
    )
    axes_flat[k].set_title(f'Solution {k+1}')
    axes_flat[k].set_xlabel('x')
    axes_flat[k].set_ylabel('y')
    plt.colorbar(im, ax=axes_flat[k], shrink=0.8)

plt.suptitle(
    f'PDE solutions on {n_grid}x{n_grid} grid (n_sf={n_sf}, vel_scale={vel_scale:.0e})',
    fontsize=14, y=1.02,
)
plt.tight_layout()
plt.show()

## 3. Velocity Fields from Streamfunctions

Each set of streamfunction coefficients defines a unique divergence-free velocity field. The velocity is evaluated analytically from the polynomial derivatives:

$$v_x = \sum a_{ij} \cdot j \cdot x^i \cdot y^{j-1}, \quad v_y = -\sum a_{ij} \cdot i \cdot x^{i-1} \cdot y^j$$

The velocity magnitude is scaled so that the root-mean-square (RMS) speed on the grid matches a prescribed target value (`vel_scale`).

In [ ]:
# Visualize the velocity field for one set of coefficients
n_vel = 32
x, y, Xc, Yc, Xef, Yef, Xnf, Ynf, h = make_grid(n_vel)

coeffs_k = C[0]
(vx_c, vy_c), (vx_f, vy_f), scaled_coeffs = eval_velocity_from_streamfunc_coeffs(
    coeffs_k, idx, Xc, Yc, Xef, Yef, Xnf, Ynf, target_rms=vel_scale,
)

speed = np.sqrt(vx_c**2 + vy_c**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Speed magnitude
im = axes[0].imshow(speed.T, origin='lower', extent=[-1, 1, -1, 1], cmap='hot')
axes[0].set_title('Velocity magnitude |v|')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Quiver plot (subsample for clarity)
step = 2
axes[1].quiver(
    Xc[::step, ::step], Yc[::step, ::step],
    vx_c[::step, ::step], vy_c[::step, ::step],
    speed[::step, ::step], cmap='hot', scale_units='xy',
)
axes[1].set_title('Velocity field (quiver)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_aspect('equal')
axes[1].set_xlim(-1, 1)
axes[1].set_ylim(-1, 1)

rms_speed = float(np.sqrt(np.mean(vx_c**2 + vy_c**2)))
plt.suptitle(f'Divergence-free velocity field (RMS speed = {rms_speed:.1f})', fontsize=13)
plt.tight_layout()
plt.show()

## 4. The Solution Manifold

Each coefficient vector $\mathbf{a} \in \mathbb{R}^{n_{\text{coeff}}}$ produces a unique PDE solution $u(\mathbf{a})$ on the grid. As we vary $\mathbf{a}$, the solutions trace out a **solution manifold** embedded in the high-dimensional space $\mathbb{R}^{n \times n}$.

Despite living in a space of dimension $n^2$ (e.g., $32^2 = 1024$), the solutions are parameterized by only $n_{\text{coeff}}$ coefficients (e.g., 14 for $n_{\text{sf}}=4$). This suggests the manifold is intrinsically low-dimensional -- exactly the structure that autoencoders can exploit.

In [ ]:
# Generate 20 solutions and visualize the manifold structure via PCA
n_samples = 20
n_grid_manifold = 32
C_manifold, idx_manifold = sample_coeffs(n_sf, n_samples, mode='uniform', seed=123)

solutions = []
for k in range(n_samples):
    u = solve_level(n_grid_manifold, C_manifold[k], idx_manifold, vel_scale)
    solutions.append(u.ravel())

U = np.array(solutions)  # (n_samples, n_grid^2)

# Compute pairwise distances
dists = np.zeros((n_samples, n_samples))
for i in range(n_samples):
    for j in range(n_samples):
        dists[i, j] = np.linalg.norm(U[i] - U[j])

# PCA via SVD (no sklearn needed)
U_centered = U - U.mean(axis=0)
_, S, Vt = np.linalg.svd(U_centered, full_matrices=False)
# Project onto first 2 principal components
pc_coords = U_centered @ Vt[:2].T

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pairwise distance matrix
im = axes[0].imshow(dists, cmap='Blues')
axes[0].set_title('Pairwise distances between solutions')
axes[0].set_xlabel('Sample index')
axes[0].set_ylabel('Sample index')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# 2D PCA projection
axes[1].scatter(pc_coords[:, 0], pc_coords[:, 1], c=np.arange(n_samples), cmap='viridis', s=80, edgecolors='k')
for k in range(n_samples):
    axes[1].annotate(str(k), (pc_coords[k, 0], pc_coords[k, 1]), fontsize=8, ha='center', va='bottom')
axes[1].set_title('PCA projection of solution manifold')
axes[1].set_xlabel('PC 1')
axes[1].set_ylabel('PC 2')
axes[1].grid(True, alpha=0.3)

# Report variance explained
var_explained = S**2 / np.sum(S**2)
print(f'Variance explained by PC1: {var_explained[0]:.1%}')
print(f'Variance explained by PC1+PC2: {sum(var_explained[:2]):.1%}')
print(f'Variance explained by first 5 PCs: {sum(var_explained[:5]):.1%}')

plt.tight_layout()
plt.show()

## 5. Autoencoder Architecture (Concept)

The full pipeline learns compact latent representations of PDE solutions using convolutional autoencoders:

1. **Encoder** $E_k$: maps a solution field $u \in \mathbb{R}^{n \times n}$ to a latent vector $z \in \mathbb{R}^d$ (e.g., $d=32$).
2. **Decoder** $D_k$: reconstructs $\hat{u} = D_k(z) \approx u$ from the latent.
3. **Latent alignment**: multiple modalities (solutions at different grid resolutions, streamfunction coefficients) are mapped into a **shared latent space** so that $E_{\text{u16}}(u_{16}) \approx E_{\text{u32}}(u_{32}) \approx \ldots \approx E_{\text{coeff}}(\mathbf{a})$ for the same sample.

**Note:** Full autoencoder training requires TensorFlow. This section shows the conceptual pipeline using SVD truncation as a linear proxy for what a nonlinear autoencoder achieves.

In [ ]:
# Demonstrate the encoding/decoding concept with SVD truncation
# (linear proxy for a nonlinear autoencoder)

# Use the solutions already generated
U_centered_demo = U - U.mean(axis=0)
_, S_demo, Vt_demo = np.linalg.svd(U_centered_demo, full_matrices=False)

# Reconstruction error as a function of latent dimension
latent_dims = list(range(1, min(n_samples, 15) + 1))
ree_values = []
for d in latent_dims:
    # Encode: project onto top-d singular vectors
    Z = U_centered_demo @ Vt_demo[:d].T  # (n_samples, d)
    # Decode: reconstruct from projection
    U_reconstructed = Z @ Vt_demo[:d] + U.mean(axis=0)
    # Relative energy error
    num = np.sum((U - U_reconstructed) ** 2)
    den = np.sum(U ** 2)
    ree_values.append(num / den)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Singular value spectrum
axes[0].semilogy(S_demo, 'bo-', markersize=6)
axes[0].set_title('Singular value spectrum')
axes[0].set_xlabel('Component index')
axes[0].set_ylabel('Singular value')
axes[0].grid(True, alpha=0.3)

# Reconstruction error vs latent dim
axes[1].semilogy(latent_dims, ree_values, 'rs-', markersize=6)
axes[1].set_title('Reconstruction error vs latent dimension')
axes[1].set_xlabel('Latent dimension d')
axes[1].set_ylabel('Relative Energy Error (REE)')
axes[1].grid(True, alpha=0.3)

# Show a reconstruction example
d_example = 5
Z_ex = U_centered_demo @ Vt_demo[:d_example].T
U_recon_ex = Z_ex @ Vt_demo[:d_example] + U.mean(axis=0)

print(f'Solution stats: mean={U.mean():.4f}, std={U.std():.4f}')
print(f'With d={d_example} components: REE = {ree_values[d_example-1]:.2e}')
print(f'Latent vector shape: {Z_ex.shape} (from solution shape {U.shape})')
print(f'Compression ratio: {U.shape[1]}/{d_example} = {U.shape[1]/d_example:.0f}x')

plt.tight_layout()
plt.show()

## 6. Cross-Modal Latent Alignment

A key innovation in this work is **cross-modal alignment**: solutions computed at different grid resolutions (e.g., $16 \times 16$ vs $32 \times 32$) represent the same underlying PDE solution at different fidelities. By aligning their latent representations, we can:

- **Encode** a coarse solution and **decode** it at a finer resolution (super-resolution).
- Transfer information between the streamfunction coefficient space and the solution field space.
- Build a **single shared latent space** that captures the essential structure across all modalities.

Below we demonstrate the low-rank structure at multiple grid resolutions, showing that the leading singular components capture most of the variance at each level.

In [ ]:
# Generate solutions at two grid levels and compare their spectral structure
n_cross = 20
levels_demo = [16, 32]
C_cross, idx_cross = sample_coeffs(n_sf, n_cross, mode='uniform', seed=77)

data_by_level = {}
for n_level in levels_demo:
    sols = []
    for k in range(n_cross):
        u = solve_level(n_level, C_cross[k], idx_cross, vel_scale)
        sols.append(u.ravel())
    data_by_level[n_level] = np.array(sols)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = {'16': 'tab:blue', '32': 'tab:orange'}
sv_data = {}

for n_level in levels_demo:
    U_level = data_by_level[n_level]
    U_level_c = U_level - U_level.mean(axis=0)
    _, S_level, _ = np.linalg.svd(U_level_c, full_matrices=False)
    sv_data[n_level] = S_level
    color = colors[str(n_level)]
    axes[0].semilogy(S_level, 'o-', color=color, markersize=5, label=f'{n_level}x{n_level}')

axes[0].set_title('Singular values by grid level')
axes[0].set_xlabel('Component index')
axes[0].set_ylabel('Singular value')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative variance explained
for n_level in levels_demo:
    S_l = sv_data[n_level]
    var_cum = np.cumsum(S_l**2) / np.sum(S_l**2)
    color = colors[str(n_level)]
    axes[1].plot(var_cum, 'o-', color=color, markersize=5, label=f'{n_level}x{n_level}')

axes[1].axhline(0.99, color='gray', linestyle='--', alpha=0.7, label='99% threshold')
axes[1].set_title('Cumulative variance explained')
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('Cumulative fraction')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Show corresponding solutions side by side
sample_idx = 0
for col_idx, n_level in enumerate(levels_demo):
    u_sample = data_by_level[n_level][sample_idx].reshape(n_level, n_level)
    ax_img = fig.add_axes([0.68 + col_idx * 0.16, 0.15, 0.14, 0.7])
    ax_img.imshow(u_sample.T, origin='lower', extent=[-1, 1, -1, 1], cmap='viridis')
    ax_img.set_title(f'{n_level}x{n_level}', fontsize=10)
    ax_img.set_xticks([])
    ax_img.set_yticks([])

# Remove the third subplot (we use manual axes instead)
axes[2].set_visible(False)

# Print summary
for n_level in levels_demo:
    S_l = sv_data[n_level]
    var_cum = np.cumsum(S_l**2) / np.sum(S_l**2)
    n_99 = int(np.searchsorted(var_cum, 0.99)) + 1
    print(f'{n_level}x{n_level}: {n_99} components needed for 99% variance (out of {len(S_l)} total)')

plt.tight_layout()
plt.show()

## 7. Summary

This notebook demonstrated the key concepts behind learning latent representations of PDE solutions:

| Modality | Description | Dimensionality |
|----------|-------------|----------------|
| `u16` | Solution on 16x16 grid | 256 |
| `u32` | Solution on 32x32 grid | 1,024 |
| `u64` | Solution on 64x64 grid | 4,096 |
| `u128` | Solution on 128x128 grid | 16,384 |
| `u256` | Solution on 256x256 grid | 65,536 |
| `coeff` | Streamfunction coefficients | 14 (for n_sf=4) |

**Key takeaways:**

1. **Low-dimensional structure**: PDE solutions, despite living in high-dimensional spaces, are parameterized by a small number of streamfunction coefficients. The singular value spectrum drops rapidly, confirming low intrinsic dimensionality.

2. **Cross-resolution consistency**: The same underlying solution at different grid resolutions shares the same low-rank structure, enabling a shared latent space across modalities.

3. **From linear to nonlinear**: SVD truncation provides a useful linear baseline. Convolutional autoencoders (trained in the full pipeline) capture additional nonlinear structure for even more compact representations.

4. **Latent alignment**: By training second-level alignment networks, all modalities can be mapped to a single shared latent space, enabling cross-modal encoding and decoding.

**For the full training pipeline** (convolutional autoencoders, latent alignment, encoder/decoder fine-tuning, and quantitative evaluation), see `demonstrator_pde.ipynb`.